# Labour productivity (GVA per hour worked / per filled job) → ITL2

Builds **labour productivity per ITL2 region per year** and saves `clean/productivity_itl2.csv`. This is the *GVA per hour worked* measure named in the RQ4 variable list; total real GVA is already in `clean/gva_itl2.csv` (`build/gva.ipynb`).

- **Source:** ONS *Subregional productivity: labour productivity (GVA per hour worked and GVA per filled job)* — `labourproductivityitls2.xlsx` (June 2025 release). **Published directly at ITL2 (2025)** — the 46 codes match the spine, so no aggregation.
- **Counts first:** we keep the two **denominators as counts** — `hours_worked_weekly` (total hours worked in an average week, 2004–2023) and `filled_jobs` (2002–2023). Both sum across regions (checked against ITL1 below), so any coarser grouping can recompute productivity exactly.

**Which productivity column to use:**

| column | what | use for |
|---|---|---|
| `gva_per_hour_gbp`, `gva_per_job_gbp` | current prices, **unsmoothed** (£) | consistent with regional GVA; year-to-year analysis |
| `..._gbp_smoothed` | current prices, **smoothed** (£) | ONS's preferred measure for **comparing levels across regions** (dampens survey noise in hours/jobs) |
| `..._real_index` | **chained volume** index, 2022 = 100, **per region** | **real productivity growth within a region**; not comparable in levels across regions |
| `..._real_2022gbp` | **real £ in 2022 prices** = 2022 current-price £ × real index / 100 (§3) | comparing **across regions and over time** together |

Current-price £ levels are *not* inflation-adjusted; the `_real_2022gbp` columns are (method in §3).

**Caveats (ONS notes):** hours are survey-based (APS/LFS-constrained), so ITL2 hours carry sampling noise — hence the smoothed series. Northern Ireland's ITL2 hours include hours that couldn't be allocated to its ITL3s (irrelevant at ITL2). GVA excludes Extra-Regio (offshore) activity.

In [1]:
import re, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

BASE  = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
RAW   = BASE / "raw" / "productivity" / "labourproductivityitls2.xlsx"
SPINE = BASE / "clean" / "geography_spine.csv"
OUT   = BASE / "clean" / "productivity_itl2.csv"
URL = ("https://www.ons.gov.uk/file?uri=/employmentandlabourmarket/peopleinwork/labourproductivity/datasets/"
       "subregionalproductivitylabourproductivitygvaperhourworkedandgvaperfilledjobindicesbyuknuts2andnuts3subregions/"
       "current/labourproductivityitls2.xlsx")

if not RAW.exists():                      # download once; delete the file to refresh to a newer release
    RAW.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    RAW.write_bytes(urllib.request.urlopen(req, timeout=120).read())
print(pd.read_excel(RAW, sheet_name="Contents", header=None).iloc[2, 0])

spine = pd.read_csv(SPINE)
itl2_name = dict(zip(spine.itl2_code, spine.itl2_name))

# sheet -> output column (all published at ITL2/ITL3; header on row 5)
SHEETS = {
    "Productivity Jobs":  "filled_jobs",
    "Productivity Hours": "hours_worked_weekly",
    "A4": "gva_per_hour_gbp",            # current price, unsmoothed, £
    "A3": "gva_per_hour_gbp_smoothed",   # current price, smoothed, £
    "A5": "gva_per_hour_real_index",     # chained volume index, 2022=100
    "B4": "gva_per_job_gbp",
    "B3": "gva_per_job_gbp_smoothed",
    "B5": "gva_per_job_real_index",
}

This workbook provides the references tables published with the ONS Subregional Productivity June 2025 release.


## 1. Extract each table at ITL1 and ITL2, reshape to long

Year columns are named `<Prefix>_<year>`; anything else (blank padding columns, footnote rows) is dropped.

In [2]:
def read_table(sheet, col, levels=("ITL1", "ITL2")):
    d = pd.read_excel(RAW, sheet_name=sheet, header=4)
    d = d[d.ITL_level.isin(levels)]
    years = [c for c in d.columns if re.fullmatch(r"[A-Za-z]+_\d{4}", str(c))]
    long = d.melt(id_vars=["ITL_level", "ITL_code"], value_vars=years, var_name="year", value_name=col)
    long["year"] = long.year.str[-4:].astype(int)
    long[col] = pd.to_numeric(long[col], errors="coerce")
    return long.set_index(["ITL_level", "ITL_code", "year"])[col]

tables = pd.concat([read_table(s, c) for s, c in SHEETS.items()], axis=1).reset_index()
for c in SHEETS.values():
    s = tables.loc[tables.ITL_level == "ITL2", ["year", c]].dropna()
    print(f"{c:28s} ITL2 {s.year.min()}-{s.year.max()}")

filled_jobs                  ITL2 2002-2023
hours_worked_weekly          ITL2 2004-2023
gva_per_hour_gbp             ITL2 2004-2023
gva_per_hour_gbp_smoothed    ITL2 2004-2023
gva_per_hour_real_index      ITL2 2004-2023
gva_per_job_gbp              ITL2 2002-2023
gva_per_job_gbp_smoothed     ITL2 2002-2023
gva_per_job_real_index       ITL2 2002-2023


## 2. Checks

1. ITL2 codes match the spine (46).
2. Counts are additive: ITL2 hours and jobs sum to their ITL1 totals.
3. Consistency: £ per hour × annual hours ≈ £ per job × jobs (both = current-price GVA), and implied average weekly hours per job is plausible (~28–34).

In [3]:
itl2 = tables[tables.ITL_level == "ITL2"].drop(columns="ITL_level").rename(columns={"ITL_code": "itl2_code"})
itl1 = tables[tables.ITL_level == "ITL1"].drop(columns="ITL_level").rename(columns={"ITL_code": "itl1_code"})
assert set(itl2.itl2_code) == set(spine.itl2_code), "ITL2 codes differ from the spine"
print(f"ITL2 codes match the spine: {itl2.itl2_code.nunique()} regions")

for c in ["hours_worked_weekly", "filled_jobs"]:
    s = itl2.assign(itl1_code=itl2.itl2_code.str[:3]).groupby(["itl1_code", "year"])[c].sum(min_count=1)
    gap = (s / itl1.set_index(["itl1_code", "year"])[c] - 1).abs().dropna()
    print(f"{c:22s} sum of ITL2 vs ITL1: max |gap| {100 * gap.max():.4f}%")

chk = itl2.dropna(subset=["gva_per_hour_gbp", "hours_worked_weekly", "gva_per_job_gbp", "filled_jobs"])
gva_h = chk.gva_per_hour_gbp * chk.hours_worked_weekly * 52
gva_j = chk.gva_per_job_gbp * chk.filled_jobs
print(f"GVA implied by hours vs by jobs: median |gap| {100 * (gva_h / gva_j - 1).abs().median():.2f}%, max {100 * (gva_h / gva_j - 1).abs().max():.2f}%")
hpj = chk.hours_worked_weekly / chk.filled_jobs
print(f"Weekly hours per filled job: min {hpj.min():.1f}, median {hpj.median():.1f}, max {hpj.max():.1f}")

ITL2 codes match the spine: 46 regions
hours_worked_weekly    sum of ITL2 vs ITL1: max |gap| 0.0000%
filled_jobs            sum of ITL2 vs ITL1: max |gap| 0.0000%
GVA implied by hours vs by jobs: median |gap| 0.01%, max 0.02%
Weekly hours per filled job: min 24.4, median 30.2, max 35.5


## 3. Real (inflation-adjusted) productivity, in 2022 prices

**Why:** the £ columns above are in *current prices* — each year in that year's pounds — so part of any rise over time is just inflation. For change over time we also want *real* values: every year expressed in the **same year's prices (2022)**.

**How (step by step):**
1. ONS already publishes real productivity for each region as a *chained volume index* (`..._real_index`, 2022 = 100). To build it, ONS takes each region's GVA **industry by industry**, removes price changes using that industry's national price deflator, adds the industries back up (re-weighting the mix year by year, "chaining"), and divides by hours worked (or filled jobs). Because it's deflated industry by industry, each region's inflation adjustment reflects **its own industry mix**.
2. The index only shows **growth within a region** (e.g. 95 → 100 = 5% real growth); it doesn't say whether one region is more productive than another.
3. To turn it into pounds we anchor it to the region's actual **2022 current-price level** (unsmoothed, the same basis as the index):

   `real £ (region, year) = current-price £ (region, 2022) × real index (region, year) / 100`

   So in 2022 real = current-price by construction; in other years the real value moves with the region's real growth.

**Result:** `gva_per_hour_real_2022gbp` and `gva_per_job_real_2022gbp` — comparable **across regions and over time**, in 2022 prices.

**Caveats:**
- Built from the **unsmoothed** series (ONS publishes no smoothed real index), so it is a little noisier year to year than `..._gbp_smoothed`.
- When ONS rebases (e.g. to 2023 = 100) in a future release, this rebuilds automatically: the code picks the year in which every region's index equals 100.
- Real values are **not additive** across regions (chain-linking); recompute ratios from components rather than summing real values.
- Cross-check below: these should line up with our real GVA (`clean/gva_itl2.csv`, chained volume in 2022 money) divided by annual hours — apart from differences between the two ONS release vintages.

In [4]:
# reference year = the year in which every region's real index equals 100
ref = [y for y, g in itl2.groupby("year") if g.gva_per_hour_real_index.dropna().round(6).eq(100).all()
                                         and g.gva_per_job_real_index.dropna().round(6).eq(100).all()]
assert len(ref) == 1, f"can't identify a single index reference year: {ref}"
REF = ref[0]
print(f"Real-index reference year: {REF} (index = 100 for all regions)")

for kind in ["hour", "job"]:
    anchor = itl2.loc[itl2.year == REF].set_index("itl2_code")[f"gva_per_{kind}_gbp"]          # current-price £ in REF
    itl2[f"gva_per_{kind}_real_{REF}gbp"] = itl2.itl2_code.map(anchor) * itl2[f"gva_per_{kind}_real_index"] / 100

# cross-check: real GVA (gva_itl2.csv, chained volume, 2022 money) / annual hours
gva = pd.read_csv(BASE / "clean" / "gva_itl2.csv")
x = itl2.merge(gva, on=["itl2_code", "year"]).dropna(subset=[f"gva_per_hour_real_{REF}gbp", "hours_worked_weekly"])
alt_real = x.gva_real_gbp_m * 1e6 / (x.hours_worked_weekly * 52)
gap = 100 * (x[f"gva_per_hour_real_{REF}gbp"] / alt_real - 1)
print(f"vs gva_itl2.csv / hours: median |gap| {gap.abs().median():.2f}%, 90th pct {gap.abs().quantile(.9):.2f}%, max {gap.abs().max():.2f}% "
      f"({len(x)} region-years; differences = different ONS release vintages)")

uk_like = itl2.groupby("year")[[f"gva_per_hour_gbp", f"gva_per_hour_real_{REF}gbp"]].median()
print("\nMedian region, GVA per hour: current-price £ vs real 2022 £")
print(uk_like.loc[[2004, 2008, 2015, 2019, 2023]].round(1).T.to_string())

Real-index reference year: 2022 (index = 100 for all regions)
vs gva_itl2.csv / hours: median |gap| 0.01%, 90th pct 0.01%, max 0.02% (920 region-years; differences = different ONS release vintages)

Median region, GVA per hour: current-price £ vs real 2022 £
year                       2004  2008  2015  2019  2023
gva_per_hour_gbp           22.6  26.1  29.7  32.7  40.1
gva_per_hour_real_2022gbp  33.1  34.3  35.5  35.8  37.0


## 4. Save

In [5]:
prod = itl2.copy()
prod.insert(1, "itl2_name", prod.itl2_code.map(itl2_name))
prod = prod[["itl2_code", "itl2_name", "year"] + list(SHEETS.values()) + [f"gva_per_hour_real_{REF}gbp", f"gva_per_job_real_{REF}gbp"]].sort_values(["year", "itl2_code"]).reset_index(drop=True)
prod.to_csv(OUT, index=False)
print(f"Saved -> {OUT}  ({len(prod)} rows, {prod.itl2_code.nunique()} regions, {prod.year.min()}-{prod.year.max()})")

latest = prod[prod.year == prod.year.max()]
cols = ["itl2_name", "gva_per_hour_gbp_smoothed", "gva_per_job_gbp_smoothed"]
print(f"\nHighest / lowest GVA per hour worked (smoothed, £), {prod.year.max()}:")
print(pd.concat([latest.nlargest(3, cols[1]), latest.nsmallest(3, cols[1])])[cols].round(1).to_string(index=False))
prod.head()

Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/productivity_itl2.csv  (1012 rows, 46 regions, 2002-2023)

Highest / lowest GVA per hour worked (smoothed, £), 2023:
                   itl2_name  gva_per_hour_gbp_smoothed  gva_per_job_gbp_smoothed
         Inner London - West                       64.6                  114335.0
         Inner London - East                       50.9                   85340.0
       East Central Scotland                       50.4                   77283.0
    Mid and South West Wales                       32.2                   49617.0
Cornwall and Isles of Scilly                       32.7                   50265.0
Shropshire and Staffordshire                       33.4                   54138.0


,itl2_code,itl2_name,year,filled_jobs,hours_worked_weekly,gva_per_hour_gbp,gva_per_hour_gbp_smoothed,gva_per_hour_real_index,gva_per_job_gbp,gva_per_job_gbp_smoothed,gva_per_job_real_index,gva_per_hour_real_2022gbp,gva_per_job_real_2022gbp
0,TLC3,Tees Valley,2002,2.731281e+05,NaN,NaN,NaN,NaN,31666.0,32700.0,78.44,NaN,43043.1656
1,TLC4,"Northumberland, Durham and Tyne & Wear",2002,7.910939e+05,NaN,NaN,NaN,NaN,31458.0,32634.0,84.31,NaN,45946.4207
2,TLD1,Cumbria,2002,2.206776e+05,NaN,NaN,NaN,NaN,32405.0,32017.0,89.21,NaN,48679.2207
3,TLD3,Greater Manchester,2002,1.215438e+06,NaN,NaN,NaN,NaN,32727.0,33491.0,76.58,NaN,47118.9082
4,TLD4,Lancashire,2002,6.498279e+05,NaN,NaN,NaN,NaN,31425.0,33246.0,83.13,NaN,47754.8598


## 5. Same measures on ITL2 **2021** boundaries (41 regions)

For joining with data on the older ITL2 2021 map (e.g. the job-flows file). ONS publishes this release on 2025 boundaries only, but it also gives **ITL3 (2025)**, and 181 of the 182 ITL3s sit wholly inside one ITL2 2021 region (checked below via the LAU1-level lookups). So each 2021 region is rebuilt from its ITL3s:

1. per ITL3: GVA (current £) = £ per filled job × filled jobs; real GVA (2022 £) = real £ per job × jobs (real level anchored as in §3); hours and jobs as published;
2. sum to ITL2 2021; recompute £ per hour / per job and real index (2022 = 100) from the sums. Smoothed £ per hour/job = hours- (jobs-) weighted average of the ITL3 smoothed values.
3. **Regions unchanged between 2021 and 2025** (same ITL3s on both maps — 31 of 41) take **ONS's published ITL2 values directly** (exact). Only the 10 changed regions use the rebuild.

**One approximation:** ITL3 *Mid Wales* (TLL41 = Ceredigion + Powys) is split across 2021 regions (Ceredigion → West Wales TLL1, Powys → East Wales TLL2). It is apportioned by each LA's share of the 16–64 population (Nomis mid-year estimates, same year) — affecting TLL1 and TLL2 only.

**Validation:** for the 31 unchanged regions, the rebuild is compared with ONS's published values — this measures the rebuild's error for the 10 changed regions. Jobs and current-price £ reproduce exactly; real (chain-linked) values are not additive, so the rebuild is approximate (median ~0.2%, max ~4%); hours differ only for NI (ONS allocates some NI hours to ITL2 but not ITL3), which is an unchanged region and so uses the published value.

In [6]:
DIR21 = BASE / "clean" / "itl2_2021"
LU25 = BASE / "raw" / "geography" / "LAD_(April_2025)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2025)_Lookup_in_the_UK.csv"
LU21 = BASE / "raw" / "geography" / "LAD_(April_2023)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2021)_Lookup_in_the_UK.csv"
POP_AGE = sorted((BASE / "raw" / "population").glob("nomis_population_age_*.csv"))[-1]     # from build/age.ipynb
s21 = pd.read_csv(DIR21 / "geography_spine.csv")
name21 = dict(zip(s21.itl2_code, s21.itl2_name))

# ITL3 (2025) -> ITL2 (2021), via LAU1; LAU1s renamed since 2021 fall back to their LA's 2021 region
l25 = pd.read_csv(LU25, encoding="utf-8-sig"); l21 = pd.read_csv(LU21, dtype=str)
m = l25.merge(l21[["LAU121CD", "ITL221CD"]], left_on="LAU125CD", right_on="LAU121CD", how="left")
m["ITL221CD"] = m.ITL221CD.fillna(m.LAD25CD.map(dict(zip(s21.la_code, s21.itl2_code))))
nest = m.groupby("ITL325CD").ITL221CD.nunique()
straddle = list(nest[nest > 1].index)
print(f"ITL3 (2025): {len(nest)} | inside one ITL2 (2021): {(nest == 1).sum()} | straddling: {straddle}")
assert straddle == ["TLL41"], "unexpected straddling ITL3 - review apportionment"

# weights: one row per (ITL3, ITL2 2021, year); straddler split by 16-64 population of its LAs
pa = pd.read_csv(POP_AGE)
pa = pa[pa.C_AGE.between(117, 165)]                                  # c_age 117..165 = ages 16..64
w1664 = pa.groupby(["GEOGRAPHY_CODE", "DATE"]).OBS_VALUE.sum()
tll41 = m[m.ITL325CD == "TLL41"][["LAD25CD", "ITL221CD"]].drop_duplicates()
wt = (w1664.unstack(0)[tll41.LAD25CD.tolist()]
           .rename(columns=dict(zip(tll41.LAD25CD, tll41.ITL221CD))).T.groupby(level=0).sum().T)
wt = (wt.div(wt.sum(axis=1), axis=0).stack().rename("w").rename_axis(["year", "itl2_2021"]).reset_index())
print("Mid Wales split (share to each 2021 region), 2023:", wt[wt.year == 2023].set_index("itl2_2021").w.round(3).to_dict())

simple = m[~m.ITL325CD.isin(straddle)].groupby("ITL325CD").ITL221CD.first()

# ITL3 tables -> GVA components per ITL3
i3 = pd.concat([read_table(s, c, levels=("ITL3",)) for s, c in SHEETS.items()], axis=1).reset_index()
i3 = i3.rename(columns={"ITL_code": "itl3"}).drop(columns="ITL_level")
ref3 = i3[i3.year == REF].set_index("itl3")
i3["gva_gbp"]       = i3.gva_per_job_gbp * i3.filled_jobs                                    # current prices
i3["gva_real_gbp"]  = i3.itl3.map(ref3.gva_per_job_gbp) * i3.gva_per_job_real_index / 100 * i3.filled_jobs
i3["sm_hour_x_h"]   = i3.gva_per_hour_gbp_smoothed * i3.hours_worked_weekly
i3["sm_job_x_j"]    = i3.gva_per_job_gbp_smoothed * i3.filled_jobs

# allocate ITL3 -> ITL2 2021 (weight 1, or Mid Wales split weights)
a = i3[i3.itl3.isin(simple.index)].assign(itl2_code=lambda d: d.itl3.map(simple), w=1.0)
b = i3[i3.itl3.isin(straddle)].merge(wt.rename(columns={"itl2_2021": "itl2_code"}), on="year")
alloc = pd.concat([a, b], ignore_index=True)
SUMS = ["filled_jobs", "hours_worked_weekly", "gva_gbp", "gva_real_gbp", "sm_hour_x_h", "sm_job_x_j"]
for c in SUMS:
    alloc[c] = alloc[c] * alloc.w
g = alloc.groupby(["itl2_code", "year"])[SUMS].sum(min_count=1).reset_index()

p21 = pd.DataFrame({"itl2_code": g.itl2_code, "year": g.year,
                    "filled_jobs": g.filled_jobs, "hours_worked_weekly": g.hours_worked_weekly})
p21["gva_per_hour_gbp"]          = g.gva_gbp / (g.hours_worked_weekly * 52)
p21["gva_per_hour_gbp_smoothed"] = g.sm_hour_x_h / g.hours_worked_weekly
p21["gva_per_job_gbp"]           = g.gva_gbp / g.filled_jobs
p21["gva_per_job_gbp_smoothed"]  = g.sm_job_x_j / g.filled_jobs
p21[f"gva_per_hour_real_{REF}gbp"] = g.gva_real_gbp / (g.hours_worked_weekly * 52)
p21[f"gva_per_job_real_{REF}gbp"]  = g.gva_real_gbp / g.filled_jobs
for kind in ["hour", "job"]:
    base = p21[p21.year == REF].set_index("itl2_code")[f"gva_per_{kind}_real_{REF}gbp"]
    p21[f"gva_per_{kind}_real_index"] = 100 * p21[f"gva_per_{kind}_real_{REF}gbp"] / p21.itl2_code.map(base)
p21.insert(1, "itl2_name", p21.itl2_code.map(name21))
p21 = p21[["itl2_code", "itl2_name", "year"] + [c for c in prod.columns if c not in ("itl2_code", "itl2_name", "year")]]
p21 = p21.sort_values(["year", "itl2_code"]).reset_index(drop=True)
assert p21.itl2_code.nunique() == 41

# unchanged regions: same set of ITL3s on the 2021 and 2025 maps
itl3_21 = m.groupby("ITL221CD").ITL325CD.apply(frozenset); itl3_25 = m.groupby("ITL225CD").ITL325CD.apply(frozenset)
same = sorted(c for c in itl3_21.index if c in itl3_25.index and itl3_21[c] == itl3_25[c] and c not in ("TLL1", "TLL2"))
changed = sorted(set(p21.itl2_code) - set(same))
print(f"Unchanged regions (published values used): {len(same)} | rebuilt from ITL3: {len(changed)} {changed}")

# validation of the rebuild on the unchanged regions
v = p21[p21.itl2_code.isin(same)].merge(prod, on=["itl2_code", "year"], suffixes=("", "_ons"))
print(f"Validation on {len(same)} unchanged regions (max |gap| %, rebuilt vs ONS ITL2):")
for c in ["filled_jobs", "hours_worked_weekly", "gva_per_hour_gbp", "gva_per_hour_gbp_smoothed",
          f"gva_per_hour_real_{REF}gbp", "gva_per_job_gbp", f"gva_per_job_real_{REF}gbp"]:
    gap = (100 * (v[c] / v[f"{c}_ons"] - 1)).abs()
    print(f"  {c:28s} median {gap.median():.3f}%, max {gap.max():.3f}%")

pub = prod[prod.itl2_code.isin(same)].assign(itl2_name=lambda d: d.itl2_code.map(name21))[p21.columns]
p21 = (pd.concat([pub, p21[p21.itl2_code.isin(changed)]], ignore_index=True)
         .assign(source=lambda d: np.where(d.itl2_code.isin(same), "ONS published ITL2", "rebuilt from ITL3 (2025)"))
         .sort_values(["year", "itl2_code"]).reset_index(drop=True))
assert p21.itl2_code.nunique() == 41 and not p21.duplicated(["itl2_code", "year"]).any()
p21.to_csv(DIR21 / "productivity_itl2.csv", index=False)
print(f"\nSaved -> {DIR21 / 'productivity_itl2.csv'}  ({len(p21)} rows, {p21.itl2_code.nunique()} regions, {p21.year.min()}-{p21.year.max()})")

ITL3 (2025): 182 | inside one ITL2 (2021): 181 | straddling: ['TLL41']


Mid Wales split (share to each 2021 region), 2023: {'TLL1': 0.366, 'TLL2': 0.634}


Unchanged regions (published values used): 31 | rebuilt from ITL3: 10 ['TLC1', 'TLC2', 'TLH1', 'TLK1', 'TLK2', 'TLL1', 'TLL2', 'TLM6', 'TLM7', 'TLM8']
Validation on 31 unchanged regions (max |gap| %, rebuilt vs ONS ITL2):
  filled_jobs                  median 0.000%, max 0.000%
  hours_worked_weekly          median 0.000%, max 6.626%
  gva_per_hour_gbp             median 0.008%, max 7.105%
  gva_per_hour_gbp_smoothed    median 0.074%, max 6.737%
  gva_per_hour_real_2022gbp    median 0.235%, max 7.271%
  gva_per_job_gbp              median 0.001%, max 0.015%
  gva_per_job_real_2022gbp     median 0.236%, max 3.802%

Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/itl2_2021/productivity_itl2.csv  (902 rows, 41 regions, 2002-2023)
